In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
from functools import reduce
print('pandas:', pd.__version__, ' polars:', pl.__version__)


In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- mexca_data_delete_filename_time_col_migration ---

# --- mexca_data_merge_audio_text_features_migration ---
FIX_MEXCA_AUDIO_ANNOTATION_PD = pd.DataFrame({
    "frame": [1, 2],
    "segment_speaker_label": ["S1", "S2"],
    "audio_energy": [0.5, 0.7],
})
FIX_MEXCA_TEXT_FEATURES_PD = pd.DataFrame({
    "frame": [1, 2],
    "segment_speaker_label": ["S1", "S2"],
    "span_text": ["hello", "world"],
    "word_count": [1, 1],
})
FIX_MEXCA_SENTIMENT_PD = pd.DataFrame({
    "frame": [1, 2],
    "span_text": ["hello", "world"],
    "sentiment": [0.1, -0.2],
})
FIX_MEXCA_DATA_MERGE_AUDIO_TEXT_FEATURES_MIGRATION_AUDIO_ANNOTATION_DICT = FIX_MEXCA_AUDIO_ANNOTATION_PD.to_dict(orient="list")
FIX_MEXCA_DATA_MERGE_AUDIO_TEXT_FEATURES_MIGRATION_TEXT_FEATURES_DICT = FIX_MEXCA_TEXT_FEATURES_PD.to_dict(orient="list")
FIX_MEXCA_DATA_MERGE_AUDIO_TEXT_FEATURES_MIGRATION_SENTIMENT_DICT = FIX_MEXCA_SENTIMENT_PD.to_dict(orient="list")

# --- mexca_data_merge_features_migration ---

# --- mexca_data_merge_video_annotation_migration ---

# --- mexca_data_merge_voice_features_migration ---

print("✅ Fixtures loaded")


In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_mexca_data_delete_filename_time_col_migration():
    @staticmethod
    def _delete_filename_time_col(df: pd.DataFrame) -> pd.DataFrame:
        if "time" in df.columns:
            del df["time"]
        if "filename" in df.columns:
            del df["filename"]
        return df
    return _delete_filename_time_col

def before_mexca_data_merge_audio_text_features_migration(audio_annotation_dict, sentiment_dict, text_features_dict):
    def _merge_audio_text_features(self, data_frames: List[pd.DataFrame]):
        # ...
        audio_text_features_df = pd.DataFrame(audio_annotation_dict)
        if self.transcription and self.transcription.segments:
            text_features_df = pd.DataFrame(text_features_dict)
            if self.sentiment and self.sentiment.segments:
                text_features_df = text_features_df.merge(
                    pd.DataFrame(sentiment_dict),
                    on=["frame", "span_text"],
                    how="left",
                )

            audio_text_features_df = audio_text_features_df.merge(
                text_features_df,
                on=["frame", "segment_speaker_label"],
                how="left",
            )
        data_frames.append(audio_text_features_df)
    return _merge_audio_text_features

def before_mexca_data_merge_features_migration():
    def merge_features(self) -> pd.DataFrame:
        # Harness context restored from the original enclosing method.
        dfs = []
        self._merge_video_annotation(data_frames=dfs)
        self._merge_audio_text_features(data_frames=dfs)
        self._merge_voice_features(data_frames=dfs)
        if len(dfs) > 0:
            dfs = map(self._delete_filename_time_col, dfs)
            self.features = reduce(
                lambda left, right: pd.merge(
                    left, right, on=["frame"], how="left"
                ),
                dfs,
            )

            time = self.features.frame * (1 / self.fps)

            self.features.insert(0, "filename", self.filename)
            self.features.insert(1, "time", time)
        return self.features
    return merge_features

def before_mexca_data_merge_video_annotation_migration():
    def _merge_video_annotation(self, data_frames: List[pd.DataFrame]):
        if self.video_annotation:
            video_annotation_dict = self.video_annotation.model_dump()
            del video_annotation_dict["face_average_embeddings"]
            data_frames.append(pd.DataFrame(video_annotation_dict))
    return _merge_video_annotation

def before_mexca_data_merge_voice_features_migration():
    def _merge_voice_features(self, data_frames: List):
        if self.voice_features:
            data_frames.append(pd.DataFrame(self.voice_features.model_dump()))
    return _merge_voice_features


In [ ]:
# ── Generated wrappers (experiment-generated Polars) ─────────────────────────

def gen_mexca_data_delete_filename_time_col_migration():
    @staticmethod
    def _delete_filename_time_col(df: pl.DataFrame) -> pl.DataFrame:
        cols = [col for col in ["time", "filename"] if col in df.columns]
        if cols:
            df = df.drop(cols)
        return df
    return _delete_filename_time_col

def gen_mexca_data_merge_audio_text_features_migration(audio_annotation_dict, sentiment_dict, text_features_dict):
    def _merge_audio_text_features(self, data_frames: List[pl.DataFrame]):
        # ...
        audio_text_features_df = pl.DataFrame(audio_annotation_dict)
        if self.transcription and self.transcription.segments:
            text_features_df = pl.DataFrame(text_features_dict)
            if self.sentiment and self.sentiment.segments:
                text_features_df = text_features_df.join(
                    pl.DataFrame(sentiment_dict),
                    on=["frame", "span_text"],
                    how="left",
                )

            audio_text_features_df = audio_text_features_df.join(
                text_features_df,
                on=["frame", "segment_speaker_label"],
                how="left",
            )
        data_frames.append(audio_text_features_df)
    return _merge_audio_text_features

def gen_mexca_data_merge_features_migration():
    from functools import reduce


    def merge_features(self):
        # ...
        if len(dfs) > 0:
            dfs = map(self._delete_filename_time_col, dfs)
            self.features = reduce(
                lambda left, right: left.join(right, on="frame", how="left"),
                dfs,
            )

            time = self.features["frame"] * (1 / self.fps)

            self.features.insert_column(0, pl.Series("filename", [self.filename] * self.features.height))
            self.features.insert_column(1, pl.Series("time", time))
        return self.features
    return merge_features

def gen_mexca_data_merge_video_annotation_migration():
    pd = pl  # LLM used `import polars as pd`
    def _merge_video_annotation(self, data_frames: List[pd.DataFrame]):
        if self.video_annotation:
            video_annotation_dict = self.video_annotation.model_dump()
            del video_annotation_dict["face_average_embeddings"]
            data_frames.append(pd.DataFrame(video_annotation_dict))
    return _merge_video_annotation

def gen_mexca_data_merge_voice_features_migration():
    def _merge_voice_features(self, data_frames: List):
        if self.voice_features:
            data_frames.append(pl.DataFrame(self.voice_features.model_dump()))
    return _merge_voice_features


In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")

def compare_l3(before_result, gen_result, label):
    left = _to_pl(before_result)
    right = _to_pl(gen_result)
    if left is None and right is None:
        print(f"⚠️  L3 skip {label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ L3 edge {label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ L3 edge {label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=False)
        print(f"✅ L3 edge {label}: MATCH")
    except Exception as e:
        print(f"❌ L3 edge {label}: MISMATCH — {e}")


In [ ]:
# === Tests: mexca_data_merge_features_migration ===

def _merge_features_subject(kind, with_frames=True):
    if kind == "before":
        frames = [pd.DataFrame({"frame": [1, 2], "filename": ["old", "old"], "video": [0.1, 0.2]}),
                  pd.DataFrame({"frame": [1, 2], "time": [9.0, 9.0], "voice": [1.0, 2.0]})]
        delete = before_mexca_data_delete_filename_time_col_migration()
    else:
        frames = [pl.DataFrame({"frame": [1, 2], "filename": ["old", "old"], "video": [0.1, 0.2]}),
                  pl.DataFrame({"frame": [1, 2], "time": [9.0, 9.0], "voice": [1.0, 2.0]})]
        delete = gen_mexca_data_delete_filename_time_col_migration()
    if not with_frames:
        frames = []
    obj = SimpleNamespace(features=None, fps=10.0, filename="sample.mp4", _delete_filename_time_col=delete)
    obj._merge_video_annotation = lambda data_frames: data_frames.extend(frames[:1])
    obj._merge_audio_text_features = lambda data_frames: data_frames.extend(frames[1:])
    obj._merge_voice_features = lambda data_frames: None
    return obj

try:
    _obj = _merge_features_subject("gen")
    _r = gen_mexca_data_merge_features_migration()(_obj)
    print("✅ L1 smoke gen_mexca_data_merge_features_migration: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_mexca_data_merge_features_migration: {type(_e).__name__}: {_e}")

try:
    _obj = _merge_features_subject("before")
    _rb = before_mexca_data_merge_features_migration()(_obj)
    print("✅ L1 smoke before_mexca_data_merge_features_migration: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_mexca_data_merge_features_migration: {type(_e).__name__}: {_e}")

try:
    _before_obj = _merge_features_subject("before")
    _gen_obj = _merge_features_subject("gen")
    _rb = before_mexca_data_merge_features_migration()(_before_obj)
    _rg = gen_mexca_data_merge_features_migration()(_gen_obj)
    compare(_rb, _rg, "mexca_data_merge_features_migration", check_row_order=True)
    compare(_before_obj.features, _gen_obj.features, "mexca_data_merge_features_migration side effect", check_row_order=True)
except Exception as _e:
    print(f"❌ L2 equivalence mexca_data_merge_features_migration: setup error — {type(_e).__name__}: {_e}")

try:
    _before_obj = _merge_features_subject("before", with_frames=False)
    _gen_obj = _merge_features_subject("gen", with_frames=False)
    _rb = before_mexca_data_merge_features_migration()(_before_obj)
    _rg = gen_mexca_data_merge_features_migration()(_gen_obj)
    if _rb is None and _rg is None and _before_obj.features is None and _gen_obj.features is None:
        print("✅ L3 edge mexca_data_merge_features_migration no component frames: MATCH")
    else:
        print(f"❌ L3 edge mexca_data_merge_features_migration no component frames: MISMATCH — before={_rb!r}, gen={_rg!r}")
except Exception as _e:
    print(f"❌ L3 edge mexca_data_merge_features_migration: {type(_e).__name__}: {_e}")
